# CRR binomial tree: convergence and early-exercise premium

Checks on `pricing.binomial.binomial_price`:

1. **Convergence**: the European tree price should approach the closed-form Black-Scholes price as the number of steps grows, with the error shrinking roughly as `1/N`.
2. **Early-exercise premium**: American - European, as a function of spot, for a put (and for a call with and without dividends).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pricing.black_scholes import black_scholes_price
from pricing.binomial import binomial_price

In [ ]:
# Market parameters (same as previous notebooks, for comparability)
S0 = 100.0
K = 100.0
T = 1.0
r = 0.05
sigma = 0.20
q = 0.02

bs_call_price = black_scholes_price(S0, K, T, r, sigma, option_type="call", q=q)
print(f"Black-Scholes reference price: {bs_call_price:.6f}")

## 1. Convergence to Black-Scholes

In [ ]:
n_steps_values = np.unique(np.logspace(1.3, 3.7, 15).astype(int))

rows = []
for n_steps in n_steps_values:
    tree_price = binomial_price(
        S0, K, T, r, sigma, option_type="call", q=q, n_steps=n_steps, exercise_style="european"
    )
    rows.append({
        "n_steps": n_steps,
        "tree_price": tree_price,
        "abs_error": abs(tree_price - bs_call_price),
    })

df = pd.DataFrame(rows)
df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(df["n_steps"], df["abs_error"], "o-", label="Binomial absolute error")

# Reference line with slope -1, anchored to the first point
C = df["abs_error"].iloc[0] * df["n_steps"].iloc[0]
ax.loglog(df["n_steps"], C / df["n_steps"], "k--", label="Slope -1 reference (1/N)")

ax.set_xlabel("Number of steps N")
ax.set_ylabel("Absolute error vs Black-Scholes")
ax.set_title("CRR binomial convergence: error vs N")
ax.legend()
fig.tight_layout()
plt.show()

## 2. Early-exercise premium: American vs. European put

In [ ]:
spot_grid = np.linspace(50, 150, 60)

american_put_prices = [
    binomial_price(S, K, T, r, sigma, option_type="put", q=q, n_steps=300, exercise_style="american")
    for S in spot_grid
]
european_put_prices = [
    binomial_price(S, K, T, r, sigma, option_type="put", q=q, n_steps=300, exercise_style="european")
    for S in spot_grid
]
premium = np.array(american_put_prices) - np.array(european_put_prices)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(spot_grid, american_put_prices, label="American put")
axes[0].plot(spot_grid, european_put_prices, "--", label="European put")
axes[0].axvline(K, color="gray", linestyle="--", linewidth=0.8, label="Strike K")
axes[0].set_xlabel("Spot price S")
axes[0].set_ylabel("Price")
axes[0].set_title("American vs. European put price")
axes[0].legend()

axes[1].plot(spot_grid, premium, color="tab:red")
axes[1].axvline(K, color="gray", linestyle="--", linewidth=0.8, label="Strike K")
axes[1].set_xlabel("Spot price S")
axes[1].set_ylabel("Early-exercise premium")
axes[1].set_title("Early-exercise premium (American - European)")
axes[1].legend()

fig.tight_layout()
plt.show()

assert np.all(premium >= -1e-8), "American put must never be worth less than the European put!"

## 3. American call: dividends are what make early exercise matter

Without dividends (`q=0`), exercising a call early forfeits the remaining time value for no benefit: American should equal European exactly, at any moneyness. With a dividend yield (`q=2%`), giving up the underlying's future payouts can make early exercise optimal for a deep in-the-money call, so a small but genuine premium appears there (at-the-money it is essentially zero, since being barely in-the-money is not enough to justify giving up the remaining time value).

In [ ]:
deep_itm_spot = 150.0

american_call_no_div = binomial_price(deep_itm_spot, K, T, r, sigma, option_type="call", q=0.0, n_steps=500, exercise_style="american")
european_call_no_div = binomial_price(deep_itm_spot, K, T, r, sigma, option_type="call", q=0.0, n_steps=500, exercise_style="european")

american_call_div = binomial_price(deep_itm_spot, K, T, r, sigma, option_type="call", q=q, n_steps=500, exercise_style="american")
european_call_div = binomial_price(deep_itm_spot, K, T, r, sigma, option_type="call", q=q, n_steps=500, exercise_style="european")

print(f"Deep ITM call (S={deep_itm_spot:.0f}, K={K:.0f}):")
print(f"  q=0%:  American = {american_call_no_div:.6f}, European = {european_call_no_div:.6f}, "
      f"premium = {american_call_no_div - european_call_no_div:.2e}")
print(f"  q={q*100:.0f}%:  American = {american_call_div:.6f}, European = {european_call_div:.6f}, "
      f"premium = {american_call_div - european_call_div:.6f}")